In [41]:
import os
import sys

learned_control = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(learned_control)
sys.path.append(os.path.join(learned_control, "simulator_lidar"))
sys.path.append(os.path.join(learned_control, "CARV"))

from pathlib import Path
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from simulator_lidar import run_simulation
from CARV.nfl_robustness_training.src.utils import mpc_unicycle
from sklearn.preprocessing import StandardScaler

In [42]:
"""
trajectory generation:
- get config files for obstacles
- sample random x,y in bounds, not too close to obstacles for start and end points with random headings
- find calculated trajectory between start and end
    - save npy files to trajectories/ directory 
    - save start and end points to start_end_points/ directory
- get associated lidar scan files for all trajectories
    - save to lidar_scans/ firectory

data processing:
- convert trajectory [lidar scans + goal distances (using trajectory and end points)] and [control inputs] to [input] [label] pairs
- split pairs into train, test, validation sets

training:
# TODO

testing:
# TODO

demonstration:
# TODO

"""

'\ntrajectory generation:\n- get config files for obstacles\n- sample random x,y in bounds, not too close to obstacles for start and end points with random headings\n- find calculated trajectory between start and end\n    - save npy files to trajectories/ directory \n    - save start and end points to start_end_points/ directory\n- get associated lidar scan files for all trajectories\n    - save to lidar_scans/ firectory\n\ndata processing:\n- convert trajectory [lidar scans + goal distances (using trajectory and end points)] and [control inputs] to [input] [label] pairs\n- split pairs into train, test, validation sets\n\ntraining:\n# TODO\n\ntesting:\n# TODO\n\ndemonstration:\n# TODO\n\n'

In [43]:
class TrajectoryGenerator():
    def __init__(self, parent_dir, traj_num=10, radii=[1.0], config_data=None):
        self.traj_num = traj_num
        self.radii = radii
        self.parent_dir = parent_dir
        self.config_dir = os.path.join(parent_dir, 'config')
        self.obstacle_config_dir = os.path.join(self.config_dir, 'obstacle_config')
        self.position_config_dir = os.path.join(self.config_dir, 'position_config')

        os.makedirs(self.config_dir, exist_ok=True)
        os.makedirs(self.obstacle_config_dir, exist_ok=True)
        os.makedirs(self.position_config_dir, exist_ok=True)

        self.config_data = config_data
        self.configs = None
        self.traj_dict = None
        self.scan_dict = None
        self.traj_ends_dict = None

    def generate_obs_config(self, overwrite=False):
        for radius in self.radii:
            filename = f'obstacle_{radius:1.3f}.yaml'
            yaml_path = os.path.join(self.obstacle_config_dir, filename)

            if os.path.exists(yaml_path) and not overwrite:
                overwrite = input(f"File {yaml_path} exists. Overwrite? [y/N] ").lower() == 'y'
                if not overwrite:
                    print("Aborting file creation")
                    return

            if self.config_data is None:
                config_data = {
                    'area': {
                        'x_min': -10,
                        'x_max': 10,
                        'y_min': -10,
                        'y_max': 10
                    },
                    'obstacles': [
                        {
                            'centroid_x': 0,
                            'centroid_y': 0,
                            'radius': radius,
                            'velocity_x': 0,
                            'velocity_y': 0,
                            'acc_x': 0,
                            'acc_y': 0,
                            'circle': True
                        }
                    ]
                }  
            else:
                config_data = self.config_data

            with open(yaml_path, 'w') as f:
                yaml.dump(config_data, f, 
                        default_flow_style=False,  # Preserve block style
                        sort_keys=False)  # Maintain order
        return
    
    def read_obs_config(self, read_all=False):
        configs = {}

        if read_all:
            for file in os.listdir(self.obstacle_config_dir):
                if file.startswith('obstacle_') and file.endswith('.yaml'):
                    try:
                        radius_str = file[len('obstacle_'):-len('.yaml')]
                        radius = float(radius_str)
                    except ValueError:
                        continue  # Skip files that don't conform to naming convention

                    yaml_path = os.path.join(self.obstacle_config_dir, file)
                    with open(yaml_path, 'r') as f:
                        configs[radius] = yaml.safe_load(f)
        else:
            for radius in self.radii:
                filename = f'obstacle_{radius:1.3f}.yaml'
                yaml_path = os.path.join(self.obstacle_config_dir, filename)

                if not os.path.exists(yaml_path):
                    raise FileNotFoundError(f"Expected config file not found: {yaml_path}")

                with open(yaml_path, 'r') as f:
                    configs[radius] = yaml.safe_load(f)

        self.configs = configs
        return
    
    def check_collision(self, obs_config, pos, eps=0.1):
        # print(f'obs_config is {obs_config}')
        # for o in obs_config['obstacles']:
        for o in obs_config:
            r = o['radius']
            x_obs = np.array([o['centroid_x'], o['centroid_y']])
            if np.linalg.norm(x_obs-pos) < r + eps:
                return True
        return False

    def generate_positions(self, obs_config, x_bounds=[-10, 10], y_bounds=[-10, 10], eps=0.1):
        # print(f'obstacle config is {obs_config}')
        thetas = []
        xs = []
        ys = []
        while len(thetas) < self.traj_num:
            theta = np.random.uniform(0, 2*np.pi)
            x = np.random.uniform(*x_bounds)
            y = np.random.uniform(*y_bounds)
            if self.check_collision(obs_config, np.array([x,y]), eps):
                continue
            else:
                thetas.append(theta)
                xs.append(x)
                ys.append(y)

        thetas = np.array(thetas)
        xs = np.array(xs)
        ys = np.array(ys)
        return np.column_stack((xs, ys, thetas))
    
    def generate_trajectories(self, x_bounds=[-10, 10], y_bounds=[-10, 10]):
        for rad, config in self.configs.items():
            obs_config = config['obstacles']
            starts = self.generate_positions(obs_config, x_bounds=x_bounds, y_bounds=y_bounds)
            goals = self.generate_positions(obs_config, x_bounds=x_bounds, y_bounds=y_bounds)
            traj_ends = np.column_stack((starts, goals))
            filename = f'traj_ends_{rad:1.3f}.npy'
            filepath = os.path.join(self.config_dir, filename)
            np.save(filepath, traj_ends)
        print(f"Saved datasets with start-goal pairs to {self.config_dir}")
        # return traj_ends_set
        return
    
    def read_traj_ends(self):
        traj_ends_dict = {}
        for rad in self.configs.keys():
            filename = f'traj_ends_{rad:1.3f}.npy'
            filepath = os.path.join(self.config_dir, filename)
            if os.path.exists(filepath):
                traj_ends = np.load(filepath)
                traj_ends_dict[rad] = traj_ends
            else:
                print(f"Warning: {filename} not found.")
        self.traj_ends_dict = traj_ends_dict
        return traj_ends_dict

    def generate_scans(self, traj_ends_dict=None):
        i = 1
        if traj_ends_dict is None:
            traj_ends_dict = self.traj_ends_dict
        for rad, config in self.configs.items():
            print(f'processing trajectory {i}')
            i += 1

            traj_ends = traj_ends_dict[rad]
            rad_str = f'{rad:1.3f}'
            for i in range(traj_ends.shape[0]):
                start = traj_ends[i,:3].tolist()
                goal = traj_ends[i,3:5].tolist()

                mpc_unicycle.generate_data(
                    num_trajectories=1, 
                    noisy=False, 
                    prob_short=0.0, 
                    use_config=True, 
                    environment=f'/obstacle_{rad_str}', 
                    # config_path=os.path.join(learned_control, 'safety_verification/config/obstacle_config'),
                    config_path=self.obstacle_config_dir,
                    save_parent_dir=os.path.join(self.parent_dir, 'trajectories'),
                    save_filename=f'mpc_traj_{rad_str}_{i}.pkl',
                    plot=True,
                    start=start,
                    goal=goal,
                )
                run_simulation.main(
                    env = f'sim_lidar_rad_{rad_str}_{i}', 
                    out_fn = f'sim_lidar_rad_{rad_str}_{i}', 
                    # out_fn = f'', 
                    out_file_type = 'carmen', 
                    save_all_data_as_npz = False,
                    n_reflections = 24, 
                    fov= 360, 
                    max_laser_distance = 12,
                    unoccupied_points_per_meter= 0.5, 
                    pose_filepath=os.path.join(self.parent_dir, f'trajectories/mpc_traj_{rad_str}_{i}.pkl'),
                    data_gen_obs_path=os.path.join(self.obstacle_config_dir, f'obstacle_{rad_str}.yaml'),
                    data_gen_out_path=os.path.join(self.parent_dir, 'lidar_scans')
                
                )
        return

    def load_trajectories(self):
        traj_dict = {}
        for rad in self.configs.keys():
            rad_str = f'{rad:1.3f}'
            traj_list = []
            i = 0
            while True:
                filename = f'mpc_traj_{rad_str}_{i}.pkl'
                filepath = os.path.join(self.parent_dir, 'trajectories', filename)
                if not os.path.exists(filepath):
                    break
                with open(filepath, 'rb') as f:
                    traj = pickle.load(f)
                    traj_list.append(traj)
                i += 1
            if traj_list:
                traj_dict[rad] = traj_list
        self.traj_dict = traj_dict
        return traj_dict
    
    def parse_gfs_lidar(self, file_path):
        scans = []
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('FLASER'):
                    parts = line.strip().split()
                    ranges = list(map(float, parts[2:26]))  # Adjust if more readings exist
                    # ranges = [max_distance if x == 12.0 else x for x in ranges]
                    scans.append(ranges)
        return np.array(scans)

    def load_scans(self, max_distance=12.0):
        scan_dict = {}
        for rad in self.configs.keys():
            rad_str = f'{rad:1.3f}'
            scans = []
            i = 0
            while True:
                # filename = f'sim_lidar_rad_{rad_str}_{i}.gfs.log'
                filename = f'sim_lidar_rad_{rad_str}_{i}.csv'
                filepath = os.path.join(self.parent_dir, 'lidar_scans', filename)
                # print(f'looking for {filepath}')
                if not os.path.exists(filepath):
                    break
                # print(f'parsing with filepath {filepath}')
                scan = self.parse_gfs_lidar(filepath)
                scans.append(scan)
                i += 1
            if scans:
                scan_dict[rad] = scans
        self.scan_dict = scan_dict
        return scan_dict
    

    def angle_normalize(self, angle):
        return (angle + np.pi) % (2 * np.pi) - np.pi

    def process_data(self):
        """
        input-output pairs of ([lidar_scan + heading_to_goal], control)
        """
        pair_dict = {}
        for rad, config in self.configs.items():
            pairs = []
            for i in range(len(self.scan_dict[rad])):
                traj, control = self.traj_dict[rad][i]
                traj_ends = self.traj_ends_dict[rad][i]
                scan = self.scan_dict[rad][i]

                goal_x, goal_y = traj_ends[3:5] 
                dx = goal_x - traj[:, 0]
                dy = goal_y - traj[:, 1]
                goal_headings = self.angle_normalize(np.arctan2(dy, dx) - traj[:,2])
                goal_headings = goal_headings.reshape(-1, 1)

                n = min([traj.shape[0], control.shape[0], scan.shape[0]])
                # input_data = np.concatenate((scan, goal_headings), axis=1)
                input_data = np.concatenate((scan[:n], goal_headings[:n]), axis=1)

                # pairs.append((input_data, control))
                pairs.append((input_data, control[:n]))

            pair_dict[rad] = pairs
        
        self.pair_dict = pair_dict
        return pair_dict

    def get_training_pairs(self, x_bounds=[-10, 10], y_bounds=[-10, 10], read=False):
        self.generate_obs_config()
        self.read_obs_config()
        if not read:
            self.generate_trajectories(x_bounds=x_bounds, y_bounds=y_bounds)
        # traj_ends_dict = self.read_traj_ends()
        # self.generate_scans(traj_ends_dict)
        self.read_traj_ends()
        self.load_scans()
        self.load_trajectories()
        if not read:
            self.generate_scans()
        # traj_dict = self.load_trajectories()
        # scan_dict = self.load_scans()
        self.process_data()
        return self.pair_dict
    
        # print(f'traj_dict={traj_dict}') # each individual traj is [nx3 points, nx1 velocities]
        # print(f'traj_dict_shape={traj_dict[1.0][0]}')
        # print(f'scan_dict={scan_dict}')
        # print(f'the input output pairs are {self.pair_dict}')
        # print(f'the input output pair shapes are {self.pair_dict[1.0]}')

# test.generate_obs_config(1.0, True)
# test.generate_obs_config(3.0)
# test.generate_obs_config(5.125)

In [44]:
safety_dir = os.path.join(learned_control, 'safety_verification')
test = TrajectoryGenerator(parent_dir=safety_dir, traj_num=100)
# test.process_data()  # this sets test.pair_dict
# pair_dict = test.pair_dict  # You may need to modify process_data to `return` or `store` pair_dict
# pair_dict = test.get_training_pairs(x_bounds=[-5,5], y_bounds=[-5,5])
pair_dict = test.get_training_pairs(x_bounds=[-5,5], y_bounds=[-5,5], read=True)

In [45]:
class ControlNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ControlNet, self).__init__()
        self.net = nn.Sequential(
            # # model 0
            # nn.Linear(input_dim, 20),
            # nn.ReLU(),
            # # overfits with too many layers
            # nn.Linear(20, 20),
            # nn.ReLU(),
            # nn.Linear(20, 20),
            # nn.ReLU(),
            # # nn.Linear(20, 20),
            # # nn.ReLU(),
            # nn.Linear(20, output_dim)

            # # model 1
            # nn.Linear(input_dim, 64),
            # nn.ReLU(),
            # nn.Linear(64, 32),
            # nn.ReLU(),
            # nn.Linear(32, 20),
            # nn.ReLU(),
            # nn.Linear(20, output_dim)
            
            # model 2
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 20),
            nn.ReLU(),
            nn.Linear(20, output_dim)
        
        )

    def forward(self, x):
        return self.net(x)

class CNNControlNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(CNNControlNet, self).__init__()
        
        # Parameters
        self.input_dim = input_dim
        
        # CNN part: expects 1D "image" input of size (input_dim - 1)
        # # model 0
        # self.conv = nn.Sequential(
        #     nn.Conv1d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
        #     nn.ReLU(),
        #     nn.Conv1d(in_channels=8, out_channels=4, kernel_size=3, padding=1),
        #     nn.ReLU()
        # )
        
        # model 1
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=8, out_channels=4, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # Compute flattened output size after CNN
        # Input: (batch_size, 1, input_dim - 1) → Output: (batch_size, 4, input_dim - 1)

        # model 0
        # cnn_output_dim = 4 * (input_dim - 1)
        
        # model 1
        cnn_output_dim = 4 * (input_dim - 1)
        
        # Fully connected layers after concatenation
        # model 0
        # self.net = nn.Sequential(
        #     nn.Linear(cnn_output_dim + 1, 64),
        #     nn.ReLU(),
        #     nn.Linear(64, 32),
        #     nn.ReLU(),
        #     nn.Linear(32, 20),
        #     nn.ReLU(),
        #     nn.Linear(20, output_dim)
        # )

        # model 1
        self.net = nn.Sequential(
            nn.Linear(cnn_output_dim + 1, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            # nn.Linear(16, 4),
            # nn.ReLU(),
            nn.Linear(16, output_dim)
        )

    def forward(self, x):
        # x shape: (batch_size, input_dim)
        
        # Split input into CNN part and final scalar
        x_cnn = x[:, :-1]  # (batch_size, input_dim - 1)
        x_scalar = x[:, -1].unsqueeze(1)  # (batch_size, 1)
        
        # Prepare CNN input: add channel dimension → (batch_size, 1, input_dim - 1)
        x_cnn = x_cnn.unsqueeze(1)
        x_cnn_out = self.conv(x_cnn)  # (batch_size, 4, input_dim - 1)
        x_cnn_flat = x_cnn_out.view(x_cnn_out.size(0), -1)  # Flatten → (batch_size, cnn_output_dim)
        
        # Concatenate with scalar input
        x_combined = torch.cat([x_cnn_flat, x_scalar], dim=1)  # (batch_size, cnn_output_dim + 1)

        return self.net(x_combined)
    
# class HybridControlNet(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(HybridControlNet, self).__init__()
#         self.input_dim = input_dim

#         self.cnn = nn.Sequential(
#             nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Conv1d(in_channels=32, out_channels=16, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.AdaptiveAvgPool1d(1)
#         )

#         self.fc = nn.Sequential(
#             nn.Linear(16 + input_dim, 64),
#             nn.ReLU(),
#             nn.Linear(64, 32),
#             nn.ReLU(),
#             nn.Linear(32, 20),
#             nn.ReLU(),
#             nn.Linear(20, output_dim)
#         )

#     def forward(self, x):
#         if x.dim() == 2:
#             if x.shape[1] != 25 * self.input_dim:
#                 raise RuntimeError(f"Expected input of shape (batch_size, {25 * self.input_dim}), got {x.shape}")
#             x = x.view(-1, 25, self.input_dim)

#         x_cnn = x[:, :24, :]           # (batch, 24, input_dim)
#         x_extra = x[:, 24, :]          # (batch, input_dim)

#         x_cnn = x_cnn.permute(0, 2, 1) # -> (batch, input_dim, 24)
#         cnn_out = self.cnn(x_cnn).squeeze(-1)  # (batch, 16)

#         combined = torch.cat([cnn_out, x_extra], dim=1)  # (batch, 16 + input_dim)
#         return self.fc(combined)


# def prepare_dataset(pair_dict, test_ratio=0.1, val_ratio=0.1, batch_size=64):
#     # Flatten across all radii
#     inputs = []
#     outputs = []
#     for rad in pair_dict:
#         for input, control in pair_dict[rad]:
#             inputs.append(input)
#             outputs.append(control)
    
#     inputs = np.concatenate(inputs, axis=0)
#     outputs = np.concatenate(outputs, axis=0)

#     # Normalize input
#     scaler = StandardScaler()
#     inputs = scaler.fit_transform(inputs)

#     # Convert to tensors
#     inputs = torch.tensor(inputs, dtype=torch.float32)
#     outputs = torch.tensor(outputs, dtype=torch.float32)

#     # Create dataset
#     dataset = TensorDataset(inputs, outputs)
#     n_total = len(dataset)
#     n_test = int(test_ratio * n_total)
#     n_val = int(val_ratio * n_total)
#     n_train = n_total - n_test - n_val

#     train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])

#     train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
#     test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

#     return train_loader, val_loader, test_loader, scaler

def prepare_dataset(pair_dict, test_ratio=0.1, val_ratio=0.1, batch_size=64):
    # Flatten across all radii
    inputs = []
    outputs = []
    for rad in pair_dict:
        for input, control in pair_dict[rad]:
            inputs.append(input)
            outputs.append(control)

    inputs = np.concatenate(inputs, axis=0)
    outputs = np.concatenate(outputs, axis=0)

    # No normalization — use raw data
    inputs = torch.tensor(inputs, dtype=torch.float32)
    outputs = torch.tensor(outputs, dtype=torch.float32)

    # Create dataset
    dataset = TensorDataset(inputs, outputs)
    n_total = len(dataset)
    n_test = int(test_ratio * n_total)
    n_val = int(val_ratio * n_total)
    n_train = n_total - n_test - n_val

    train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, dr=0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=dr)
    best_train_loss = None
    best_val_loss = None
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_batch.size(0)

        train_loss /= len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                y_pred = model(x_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item() * x_batch.size(0)
        val_loss /= len(val_loader.dataset)
        
        if best_train_loss is None or train_loss < best_train_loss:
            best_train_loss = train_loss
        if best_val_loss is None or val_loss < best_val_loss:
            best_val_loss = val_loss

        # print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"\rEpoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}", end="")
        sys.stdout.flush()
    print()
    return model, best_train_loss, best_val_loss

In [46]:
def evaluate_model(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    criterion = nn.MSELoss()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_pred = model(x_batch)

            loss = criterion(y_pred, y_batch)
            total_loss += loss.item() * x_batch.size(0)

            all_preds.append(y_pred.cpu())
            all_targets.append(y_batch.cpu())

    avg_loss = total_loss / len(test_loader.dataset)
    print(f"Test MSE: {avg_loss:.4f}")

    # Optional: Show some predictions
    preds = torch.cat(all_preds, dim=0).numpy()
    targets = torch.cat(all_targets, dim=0).numpy()
    
    print("\nSample predictions (predicted vs. true):")
    for i in range(min(10, len(preds))):
        print(f"{preds[i]}  <==>  {targets[i]}")

    return avg_loss, preds, targets

In [47]:
import json

def save_results_dict_key(filename, lr, dr, best_train_loss, best_val_loss, test_loss):
    key = f"({lr}, {dr})" 
    
    if os.path.exists(filename):
        with open(filename, "r") as f:
            data = json.load(f)
    else:
        data = {}
    
    data[key] = {
        "best_train_loss": best_train_loss,
        "best_val_loss": best_val_loss,
        "test_loss": test_loss
    }
    
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

def find_best_combo_by_test_loss(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)

    best_lr_dr = None
    best_test_loss = float('inf')
    best_train_loss = None
    best_val_loss = float('inf')

    for key, losses in data.items():
        test_loss = losses.get("test_loss", float('inf'))
        train_loss = losses.get("best_train_loss", None)
        val_loss = losses.get("best_val_loss", None)

        # if test_loss < best_test_loss:
        if val_loss < best_val_loss:
            best_test_loss = test_loss
            best_lr_dr = eval(key)  # Caution: eval is ok here if you trust the file
            best_train_loss = train_loss
            best_val_loss = val_loss

    print(f"filepath: {filepath}")
    print(f"Best combo based on test loss:")
    print(f"lr={best_lr_dr[0]}, dr={best_lr_dr[1]}")
    print(f"Train Loss: {best_train_loss:.6f}, Validation Loss: {best_val_loss:.6f}, Test Loss: {best_test_loss:.6f}")


In [48]:
# only run this once for dataset creation
train_loader, val_loader, test_loader = prepare_dataset(pair_dict)

In [49]:
model_name = 'cnn_model_1'
results_filepath = os.path.join(safety_dir, f'results/{model_name}_results.json')

In [ ]:
save_dir = f'saved_models/{model_name}'
model_save_dir = os.path.join(safety_dir, save_dir)
os.makedirs(model_save_dir, exist_ok=True)

example_input, example_output = next(iter(train_loader))
input_dim = example_input.shape[1]
print(f'the input dimension is {input_dim}')
output_dim = example_output.shape[1]
print(f'the output dimension is {output_dim}')

# model = ControlNet(input_dim=input_dim, output_dim=output_dim)
model = CNNControlNet(input_dim=input_dim, output_dim=output_dim)
# model = HybridControlNet(input_dim=input_dim, output_dim=output_dim)

lrs = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0]
drs = [0, 1e-6, 1e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
epochs = 250
# lrs = [1e-4]
# drs = [0]
# epochs = 10

n = len(lrs)*len(drs)
i = 1
for lr in lrs:
    for dr in drs:
        print(f'combo {i}')
        i += 1
        trained_model, best_train_loss, best_val_loss = train_model(model, train_loader, val_loader, epochs=epochs, lr=lr, dr=dr)
        test_loss, predictions, targets = evaluate_model(trained_model, test_loader)

        model_filename = f'lr{lr}_dr{dr}.pt'
        model_path = os.path.join(model_save_dir, model_filename)
        torch.save(trained_model.state_dict(), model_path)

        save_results_dict_key(results_filepath, lr, dr, best_train_loss, best_val_loss, test_loss)

the input dimension is 25
the output dimension is 1
combo 1
Epoch 10/10 - Train Loss: 0.2994 | Val Loss: 0.2898
Test MSE: 0.2716

Sample predictions (predicted vs. true):
[0.08815029]  <==>  [-1.3496237e-05]
[0.13681398]  <==>  [0.9424778]
[-0.18950872]  <==>  [0.04431753]
[-0.27881843]  <==>  [-0.9424778]
[-0.32166517]  <==>  [-0.9424778]
[-0.4735846]  <==>  [-0.9424778]
[-0.11896628]  <==>  [0.0165495]
[0.07457706]  <==>  [0.7662518]
[-0.07394387]  <==>  [1.2658238e-05]
[-0.09537082]  <==>  [-0.03139609]


In [30]:
find_best_combo_by_test_loss(results_filepath)

filepath: /Users/dgaillard/Desktop/aerospace_controls_lab/learned_control/safety_verification/results/cnn_model_0_results.json
Best combo based on test loss:
lr=0.001, dr=0.0003
Train Loss: 0.029510, Validation Loss: 0.065374, Test Loss: 0.058599


In [39]:
model = CNNControlNet(input_dim=input_dim, output_dim=output_dim)
trained_model, best_train_loss, best_val_loss = train_model(model, train_loader, val_loader, epochs=epochs, lr=0.001, dr=0.0003)
test_loss, predictions, targets = evaluate_model(trained_model, test_loader)

Epoch 250/250 - Train Loss: 0.0375 | Val Loss: 0.1215
Test MSE: 0.0752

Sample predictions (predicted vs. true):
[0.5607602]  <==>  [0.94247776]
[-0.05697246]  <==>  [-0.01485892]
[0.8817622]  <==>  [0.9424778]
[-0.93342876]  <==>  [-0.9424778]
[-0.55987847]  <==>  [-0.9424777]
[-0.5730042]  <==>  [-0.9424774]
[-0.94706506]  <==>  [-0.80695486]
[-0.97796905]  <==>  [-0.9424778]
[-0.19065471]  <==>  [-6.4170745e-06]
[-0.29276663]  <==>  [0.01111354]
